# ZEUS：从试运行到 Iris 向量

先用随机输入确认官方权重能在 CPU 上推理，再提取 OpenML ID 61（Iris）的样本向量。

官方源码：[gmum/zeus](https://github.com/gmum/zeus)，提交 `3f75d280fe7edbe12fb1a80b4058b010af115e65`；权重于 2026-09-29 从其 README 中的 [Google Drive 链接](https://drive.google.com/file/d/1D7uikacymUnmmMxjUjBuCNIomqhBWS67/view?usp=sharing)下载。权重里有当前推理代码不用的 `cluster_embedding.weight`，加载时会显示这一项。

In [1]:
from pathlib import Path
import sys

import torch

root = Path.cwd().parent  # notebook 位于 notebooks/，内核也从这里启动
sys.path.insert(0, str(root / 'vendor' / 'zeus'))

from zeus.configs import GMMConfig
from zeus.initialziation import model_initialization

In [2]:
torch.manual_seed(42)
config = GMMConfig(device='cpu', dist_based_logit=True)
model = model_initialization(config)
checkpoint = torch.load(root / 'models' / 'zeus.pt', map_location='cpu', weights_only=False)
load_result = model.load_state_dict(checkpoint['model'], strict=False)
model.eval()

x = torch.randn(8, 1, config.dim)
with torch.inference_mode():
    output = model(x, k=2)

embeddings = output[:-config.num_gaussians, 0, :]
print(f'Python: {sys.version.split()[0]}, PyTorch: {torch.__version__}')
print(f'输入: {tuple(x.shape)}, 样本向量: {tuple(embeddings.shape)}')
print(f'权重轮次: {checkpoint["epoch"]}, 数值有限: {torch.isfinite(embeddings).all().item()}')
print(f'未匹配的权重: {load_result}')

Python: 3.11.9, PyTorch: 2.5.1+cpu
输入: (8, 1, 30), 样本向量: (8, 512)
权重轮次: 160, 数值有限: True
未匹配的权重: _IncompatibleKeys(missing_keys=[], unexpected_keys=['cluster_embedding.weight'])


## 提取 Iris

沿用官方 `load_real_datasets` 的数值特征填补与缩放、标签编码和样本顺序。官方 `evaluate_model` 在输入维度不足 30 时补零，并从模型输出末尾去掉 10 个簇中心；向量的第二个版本按官方 K-means 路径缩放到 `[-1, 1]`。真实标签只用于确定传给模型的簇数和保存结果。

In [3]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from zeus.datasets import load_real_datasets

openml_id = 61
features, labels, _, n_numeric = load_real_datasets(
    config.dim, openml_idx=[openml_id], return_whole_dataset=True
)[0]
n_samples, processed_dim = features.shape
n_clusters = len(torch.unique(labels))
assert processed_dim <= config.dim
model_features = torch.cat(
    (features, torch.zeros(n_samples, config.dim - processed_dim)), dim=1
)
model_input = model_features.unsqueeze(1).to(config.device)
assert torch.isfinite(model_input).all()
print(f'OpenML ID: {openml_id}; 样本: {n_samples}; 数值特征: {n_numeric}; 处理后维度: {processed_dim}')
print(f'模型输入: {tuple(model_input.shape)}; 设备: {model_input.device}; 簇数（来自标签）: {n_clusters}')

OpenML ID: 61; 样本: 150; 数值特征: 4; 处理后维度: 4
模型输入: (150, 1, 30); 设备: cpu; 簇数（来自标签）: 3


In [4]:
with torch.inference_mode():
    model_output = model(model_input, k=n_clusters)

sample_embeddings = model_output[:-config.num_gaussians, 0, :].cpu().numpy()
scaled_embeddings = MinMaxScaler(feature_range=(-1, 1)).fit_transform(sample_embeddings)
label_values = labels.cpu().numpy()
sample_indices = np.arange(n_samples)
assert sample_embeddings.shape == scaled_embeddings.shape == (n_samples, config.embed_dim)
assert np.isfinite(sample_embeddings).all() and np.isfinite(scaled_embeddings).all()

output_path = root / 'data' / 'embeddings' / f'openml_{openml_id}.npz'
output_path.parent.mkdir(parents=True, exist_ok=True)
np.savez_compressed(
    output_path, openml_id=openml_id, embeddings=sample_embeddings,
    embeddings_scaled=scaled_embeddings, labels=label_values,
    sample_indices=sample_indices, processed_features=features.numpy(),
)
print(f'样本向量: {sample_embeddings.shape}; 缩放后: {scaled_embeddings.shape}')
print(f'数值有限: {np.isfinite(sample_embeddings).all()}; 已保存: {output_path.relative_to(root)}')

样本向量: (150, 512); 缩放后: (150, 512)
数值有限: True; 已保存: data/embeddings/openml_61.npz


## K-means 基线比较

对同一批 Iris 样本分别使用官方预处理后的 4 维特征和按官方 `predict_clusters` 缩放到 `[-1, 1]` 的 512 维 ZEUS 向量。两条路径都设 `n_init=100`、`random_state=42`。簇数 3 来自真实标签；标签只用于确定簇数和计算 ARI。这里是单个种子的流程检查，同时记录 ARI 原值和论文表格常用的 `ARI × 100`。

In [5]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

seed = 42
with np.load(output_path) as saved:
    baseline_features = saved['processed_features']
    zeus_features = saved['embeddings_scaled']
    true_labels = saved['labels']
    assert np.array_equal(saved['sample_indices'], np.arange(len(true_labels)))

assert len(baseline_features) == len(zeus_features) == len(true_labels)
n_clusters = len(np.unique(true_labels))
rows = []
for method, data, preprocessing in [
    ('preprocessed features + K-means', baseline_features, 'official numeric imputation + StandardScaler + MinMaxScaler(-1, 1)'),
    ('scaled ZEUS + K-means', zeus_features, 'same input preprocessing; ZEUS; per-dimension MinMaxScaler(-1, 1)'),
]:
    predictions = KMeans(n_clusters=n_clusters, n_init=100, random_state=seed).fit_predict(data)
    ari = adjusted_rand_score(true_labels, predictions)
    rows.append({
        'openml_id': openml_id, 'method': method, 'n_samples': len(true_labels),
        'n_features': data.shape[1], 'n_clusters': n_clusters,
        'cluster_count_source': 'true labels', 'seed': seed, 'n_init': 100,
        'preprocessing': preprocessing, 'ari': ari, 'ari_x100': ari * 100,
    })

results = pd.DataFrame(rows)
results_path = root / 'results' / 'zeus_iris_baseline.csv'
results_path.parent.mkdir(parents=True, exist_ok=True)
results.to_csv(results_path, index=False)
print(results[['method', 'n_samples', 'n_features', 'n_clusters', 'seed', 'n_init', 'ari', 'ari_x100']].to_string(index=False))
print(f'已保存: {results_path.relative_to(root)}')

                         method  n_samples  n_features  n_clusters  seed  n_init      ari  ari_x100
preprocessed features + K-means        150           4           3    42     100 0.716342 71.634211
          scaled ZEUS + K-means        150         512           3    42     100 0.851457 85.145693
已保存: results/zeus_iris_baseline.csv
